# 03 — Silver (eventos + customers_orgs)

Limpieza, enriquecimiento, features y **quarantine** desde Bronze.

**Reglas activas:**
1. `event_id` no nulo y único → quarantine
2. `cost_usd_increment < -0.01` → flag `is_cost_anomaly` (permanece en Silver)
3. `unit` nulo con `value` presente → quarantine

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SILVER, QUARANTINE
from src.jobs.silver_batch import (
    CUSTOMERS_ORGS_SILVER,
    USAGE_EVENTS_SILVER,
    USAGE_EVENTS_QUARANTINE,
)

print(f"SILVER:     {SILVER}")
print(f"QUARANTINE: {QUARANTINE}")

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("silver").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [ ]:
from src.jobs.silver_batch import run_silver

results = run_silver(spark)
results

In [ ]:
import pandas as pd

events = results[1]
summary = {
    "bronze": events["raw_count"],
    "silver_valid": events["valid_count"],
    "quarantine": events["quarantine_count"],
    "cost_anomalies_flagged": events["cost_anomalies_flagged"],
}
pd.DataFrame([summary])

In [ ]:
events["quarantine_sample"]

In [ ]:
df = spark.read.parquet(USAGE_EVENTS_SILVER)
df.select(
    "event_id", "org_name", "usage_date", "service",
    "daily_cost_usd", "requests", "genai_tokens", "carbon_kg", "is_cost_anomaly"
).show(5, truncate=False)

In [ ]:
from src.jobs.silver_batch import validate_silver

validation = validate_silver(spark)
pd.DataFrame([validation])